# T1 v2 · Survivability heatmap with three fixes (CPU, no GPU)

Second version of task **T1** (`t1-survivability-heatmap.ipynb`, kept as it ran on Kaggle). Goal **O3**: the survivability heatmap is one of KR3's four required outputs.

**What changed from v1**, after checking v1's 20 overlays against the ground truth on 19 Oct:
1. **Each hot spot is marked at its strongest point, not at its middle.** In v1, 10 of the 48 circles sat between buildings, because the middle of a large zone, or of two merged zones, is often a gap.
2. **Zones smaller than 500 pixels are ignored.** v1 circled two specks of 25 and 64 pixels.
3. **Safe areas stay at least 100 px from the image edge.** In v1, 52 of the 57 safe areas sat on the edge, where damage just outside the picture is invisible.

Also new:
- **It runs on Kaggle or on a PC.** scipy and PIL replace OpenCV, so it needs no installs; the maps match v1's up to rounding. On a PC it reads E1's predictions from `E1-results/` next to this notebook and writes to `T1-results/T1-v2/`.
- **The markers are also drawn on panel 1.** When the after-image is not available (a PC without the DisasterM3 images), panel 1 shows the true damage map instead: grey background, green intact, yellow damaged, red destroyed.
- **Stricter checks:** whether each hot spot's marked point is within 20 px of a ground-truth destroyed (or damaged) building, and whether each safe area sits on a ground-truth building.

**How to run**
- **Kaggle:** CPU only, internet off. Inputs as v1: the DisasterM3 mirror and `d4-r9-val-predictions` (E1's output). Save & Run All. Outputs in `/kaggle/working/T1-v2/`.
- **PC:** run it from `Essential_task_delegation/Champion_Task/`. Outputs in `T1-results/T1-v2/`.

**Done when:** in the 20 overlays the circles sit on damage and the safe areas are off the edge, with the stricter shares reported next to v1's.

In [1]:
# ── Cell 1: Configuration and inputs (CPU only, no installs; Kaggle or a PC) ──
import glob
import json
import re
import time
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from scipy import ndimage
from tqdm.auto import tqdm

ON_KAGGLE = Path("/kaggle/input").exists()
SMOKE_TEST = False             # True = 30 images

PRED_SET = "R9_tta"            # E1 exports R9_std and R9_tta; TTA has the best Destroyed IoU
SIGMA_PX = 25                  # danger-map blur (kernel density bandwidth), as in v1
DAMAGED_WEIGHT = 0.5           # destroyed pixels weigh 1.0
SAFE_SCALE_PX = 200            # this far from any damage counts as fully safe
HOT_THRESHOLD = 0.05           # search priority above this forms a hot zone
MIN_SPOT_PX = 500              # v2: zones smaller than this are specks, not hot spots
SAFE_MIN_GAP_PX = 150          # the 3 safest areas are at least this far apart
SAFE_EDGE_MARGIN_PX = 100      # v2: safe areas stay this far from the image edge
POINT_RADIUS_PX = 20           # v2: a marked point is "on" a building if one is this close (a few metres)
DISPLAY_MAX = 0.5              # colour scale top for danger and priority, the same for every image
N_CHECK = 10                   # overlays for the top-N xBD and top-N other images
# v1 on the same 20 images, from the 19 Oct check (the same 20-pixel and 50-pixel tests)
V1_ON_20 = {"circles": 48, "point_on_destroyed": 26, "point_on_damage": 36,
            "safe_areas": 57, "safe_within_50px_of_edge": 52, "safe_on_building": 9}

if ON_KAGGLE:
    DATA_ROOT = Path("/kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct")
    OUT_DIR = Path("/kaggle/working/T1-v2")
    hits = sorted(glob.glob(f"/kaggle/input/**/{PRED_SET}", recursive=True))
else:                          # on a PC, run from Champion_Task: E1's predictions are local, the after-images are not
    DATA_ROOT = None
    OUT_DIR = Path("T1-results/T1-v2")
    hits = sorted(glob.glob(f"E1-results/**/{PRED_SET}", recursive=True))
if not hits:
    raise FileNotFoundError(f"No '{PRED_SET}' folder: on Kaggle attach d4-r9-val-predictions (E1's output); "
                            "on a PC run this notebook from Champion_Task, next to E1-results/.")
PRED_DIR = Path(hits[0])
GT_DIR = PRED_DIR.parent.parent / "gt"
names = sorted(p.name for p in PRED_DIR.glob("*.png"))
if SMOKE_TEST:
    names = names[:30]

(OUT_DIR / "maps").mkdir(parents=True, exist_ok=True)
(OUT_DIR / "overlays").mkdir(exist_ok=True)
t_start = time.time()
print(f"✓ T1 v2 survivability heatmap{'  [SMOKE TEST]' if SMOKE_TEST else ''} ({'Kaggle' if ON_KAGGLE else 'PC'})")
print(f"  Predictions:  {PRED_DIR} ({len(names)} masks)")
print(f"  Ground truth: {GT_DIR}")
print(f"  After-images: {DATA_ROOT if DATA_ROOT else 'not on this machine: panel 1 shows the true damage map'}")

✓ T1 v2 survivability heatmap (PC)
  Predictions:  E1-results\E1\preds\R9_tta (645 masks)
  Ground truth: E1-results\E1\gt
  After-images: not on this machine: panel 1 shows the true damage map


## 1. The three maps, hot spots and safe areas
- **Danger** = Gaussian blur (sigma 25 px) of 1.0 × destroyed + 0.5 × damaged: the local share of collapsed ground.
- **Safety** = min(distance to the nearest damaged or destroyed pixel / 200 px, 1).
- **Priority** = danger × (1 − safety).
- **Hot spots** are the connected zones where priority ≥ 0.05 and the zone has at least 500 pixels, ranked by their total priority. Each one is marked at its strongest point (v1: at its middle).
- **Safe areas** are the background pixels farthest from any damage, at least 150 px apart and at least 100 px from the image edge (v1: no edge rule).

All of these are settings in Cell 1.

In [2]:
# ── Cell 2: Maps, hot spots and safe areas ─────────────────────────────────
XBD_NAME = re.compile(r"^(.+?)_(\d{8})_post_disaster")
OTHER_NAME = re.compile(r"^(.+?)_post")
EIGHT = np.ones((3, 3), bool)

def event_and_source(stem):
    """Same rule as E1: 'hurricane_florence_00000261_post_disaster' -> ('hurricane_florence', 'xBD'),
    'turkey_earthquake5_post_12' -> ('turkey_earthquake', 'other')."""
    m = XBD_NAME.match(stem)
    if m:
        return m.group(1), "xBD"
    m = OTHER_NAME.match(stem)
    return re.sub(r"\d+$", "", m.group(1) if m else stem), "other"

def read_mask(path):
    """uint8 mask: 0 background, 1 intact, 2 damaged, 3 destroyed."""
    return np.array(Image.open(path))

def survivability_maps(mask):
    """Returns danger, safety and priority (float32, 0-1), and the distance in px to the nearest damage."""
    weight = (mask == 3).astype(np.float32) + DAMAGED_WEIGHT * (mask == 2)
    danger = ndimage.gaussian_filter(weight, SIGMA_PX, mode="mirror", truncate=4.0)   # = v1's cv2.GaussianBlur
    damage = mask >= 2
    dist = (ndimage.distance_transform_edt(~damage).astype(np.float32) if damage.any()
            else np.full(mask.shape, np.inf, np.float32))
    safety = np.minimum(dist / SAFE_SCALE_PX, 1).astype(np.float32)
    return danger, safety, danger * (1 - safety), dist

def compass(x, y, w, h):
    """Where a point sits in the image (top = north): 'north-east', 'west', 'centre', ..."""
    row = ("north", "", "south")[min(int(3 * y / h), 2)]
    col = ("west", "", "east")[min(int(3 * x / w), 2)]
    return "-".join(p for p in (row, col) if p) or "centre"

def hot_spots(priority, k=3):
    """The k strongest zones (connected, priority >= HOT_THRESHOLD, at least MIN_SPOT_PX pixels), ranked by
    total priority. Each is marked at its strongest point, the one nearest the zone's middle if several tie,
    so the mark stays on the damage even when a zone is large or two zones have merged.
    Also returns the label image ('label' is each zone's id in it)."""
    labels, n = ndimage.label(priority >= HOT_THRESHOLD, structure=EIGHT)
    if n == 0:
        return [], labels
    area = np.bincount(labels.ravel(), minlength=n + 1)
    mass = np.bincount(labels.ravel(), weights=priority.ravel(), minlength=n + 1)
    mass[0] = -1                                    # label 0 is everything below the threshold
    mass[area < MIN_SPOT_PX] = -1                   # specks
    h, w = priority.shape
    spots = []
    for i in np.argsort(-mass)[:k]:
        if mass[i] < 0:
            break
        ys, xs = np.nonzero(labels == i)
        values = priority[ys, xs]
        top = values >= values.max() - 1e-6
        j = np.argmin((ys[top] - ys.mean()) ** 2 + (xs[top] - xs.mean()) ** 2)
        x, y = int(xs[top][j]), int(ys[top][j])
        spots.append({"rank": len(spots) + 1, "label": int(i), "point_xy": [x, y], "area_px": int(area[i]),
                      "peak": round(float(values.max()), 3), "where": compass(x, y, w, h)})
    return spots, labels

def safe_areas(dist, mask, k=3):
    """The k open (background) points farthest from any damage, at least SAFE_MIN_GAP_PX apart and at least
    SAFE_EDGE_MARGIN_PX from the image edge, because damage just outside the picture is invisible."""
    if np.isinf(dist).all():
        return []                                   # no damage predicted: the whole image is safe
    h, w = mask.shape
    m = SAFE_EDGE_MARGIN_PX
    inner = (slice(m, h - m), slice(m, w - m))
    score = np.full(mask.shape, -1.0)
    score[inner] = np.where(mask[inner] == 0, dist[inner], -1.0)
    yy, xx = np.ogrid[:h, :w]
    areas = []
    for rank in range(1, k + 1):
        y, x = np.unravel_index(np.argmax(score), score.shape)
        if score[y, x] < 0:
            break
        areas.append({"rank": rank, "xy": [int(x), int(y)], "distance_px": int(score[y, x]),
                      "edge_px": int(min(x, y, w - 1 - x, h - 1 - y)), "where": compass(x, y, w, h)})
        score[(yy - y) ** 2 + (xx - x) ** 2 < SAFE_MIN_GAP_PX ** 2] = -1
    return areas

# Self-checks: one destroyed block -> one hot spot, marked on the block; safe areas far from it and off the
# edge; a speck is not a hot spot; no damage -> no safe areas
_m = np.zeros((600, 600), np.uint8)
_m[280:320, 280:320] = 3
_d, _s, _p, _dist = survivability_maps(_m)
_spots, _ = hot_spots(_p)
assert len(_spots) == 1 and _spots[0]["where"] == "centre"
assert _m[_spots[0]["point_xy"][1], _spots[0]["point_xy"][0]] == 3
_safe = safe_areas(_dist, _m)
assert len(_safe) == 3 and all(a["distance_px"] > 150 and a["edge_px"] >= SAFE_EDGE_MARGIN_PX for a in _safe)
_p2 = np.zeros((300, 300), np.float32)
_p2[10:20, 10:20] = 0.06                        # a 100-pixel speck
_p2[100:150, 100:150] = 0.5                     # a real zone
assert [s["area_px"] for s in hot_spots(_p2)[0]] == [2500]
assert safe_areas(np.full((5, 5), np.inf, np.float32), np.zeros((5, 5), np.uint8)) == []
print("✓ Map functions ready (self-checks passed)")

✓ Map functions ready (self-checks passed)


In [3]:
# ── Cell 3: Every validation image -> maps, hot spots, safe areas and the ground-truth checks ──
def save_map(m, path):
    """256 x 256 copy (area average), value x 255."""
    small = np.asarray(Image.fromarray(np.clip(m, 0, 1).astype(np.float32)).resize((256, 256), Image.BOX))
    Image.fromarray(np.round(small * 255).astype(np.uint8)).save(path)

def near(gt, xy, classes):
    """Is one of these ground-truth classes within POINT_RADIUS_PX of the point?"""
    x, y, r = xy[0], xy[1], POINT_RADIUS_PX
    return bool(np.isin(gt[max(0, y - r):y + r + 1, max(0, x - r):x + r + 1], classes).any())

records, rows = [], []
for name in tqdm(names, desc="Images"):
    pred, gt = read_mask(PRED_DIR / name), read_mask(GT_DIR / name)
    stem = Path(name).stem
    event, source = event_and_source(stem)
    danger, safety, priority, dist = survivability_maps(pred)
    spots, labels = hot_spots(priority)
    for s in spots:
        zone = labels == s.pop("label")
        s["on_gt_destroyed"] = bool((zone & (gt == 3)).any())          # v1's check: the zone touches it
        s["on_gt_damage"] = bool((zone & (gt >= 2)).any())
        s["point_on_gt_destroyed"] = near(gt, s["point_xy"], [3])      # v2: the marked point itself
        s["point_on_gt_damage"] = near(gt, s["point_xy"], [2, 3])
    safe = safe_areas(dist, pred)
    for a in safe:
        a["on_gt_building"] = bool(gt[a["xy"][1], a["xy"][0]] > 0)
    # the same method on the ground-truth mask: the best a perfect segmenter could do
    gt_spots, gt_labels = hot_spots(survivability_maps(gt)[2])
    gt_on_destroyed = sum(bool(((gt_labels == s["label"]) & (gt == 3)).any()) for s in gt_spots)

    records.append({"image": name, "event": event, "source": source,
                    "pred_damaged_px": int((pred == 2).sum()), "pred_destroyed_px": int((pred == 3).sum()),
                    "hot_spots": spots, "safe_areas": safe})
    rows.append({"image": name, "event": event, "source": source, "gt_destroyed_px": int((gt == 3).sum()),
                 "hot_spots": len(spots), "on_gt_destroyed": sum(s["on_gt_destroyed"] for s in spots),
                 "on_gt_damage": sum(s["on_gt_damage"] for s in spots),
                 "point_on_gt_destroyed": sum(s["point_on_gt_destroyed"] for s in spots),
                 "point_on_gt_damage": sum(s["point_on_gt_damage"] for s in spots),
                 "safe_areas": len(safe), "safe_on_gt_building": sum(a["on_gt_building"] for a in safe),
                 "gt_hot_spots": len(gt_spots), "gt_on_destroyed": gt_on_destroyed})
    for kind, m in (("kde", danger), ("safe", safety), ("priority", priority)):
        save_map(m, OUT_DIR / "maps" / f"{stem}_{kind}.png")

per_image = pd.DataFrame(rows)

def hit_rates(df):
    n = max(int(df["hot_spots"].sum()), 1)
    return {"images": len(df), "hot_spots": int(df["hot_spots"].sum()),
            "on_gt_destroyed": round(df["on_gt_destroyed"].sum() / n, 3),
            "ceiling_from_gt_mask": round(df["gt_on_destroyed"].sum() / max(int(df["gt_hot_spots"].sum()), 1), 3),
            "on_gt_damaged_or_destroyed": round(df["on_gt_damage"].sum() / n, 3),
            "point_on_gt_destroyed": round(df["point_on_gt_destroyed"].sum() / n, 3),
            "point_on_gt_damage": round(df["point_on_gt_damage"].sum() / n, 3),
            "safe_on_gt_building": round(df["safe_on_gt_building"].sum() / max(int(df["safe_areas"].sum()), 1), 3)}

# Hot spots also come from damaged-only areas, so even a perfect mask can't put all of them on destroyed
# pixels: read on_gt_destroyed against ceiling_from_gt_mask, not against 100%.
summary = pd.DataFrame([
    {"images_from": population, "source": source, **hit_rates(g)}
    for population, df in [("all", per_image), ("with destroyed", per_image[per_image["gt_destroyed_px"] > 0])]
    for source, g in [("all", df), *df.groupby("source")]])
with open(OUT_DIR / "hotspots.json", "w") as f:
    json.dump(records, f, indent=1)
summary.to_csv(OUT_DIR / "T1_summary.csv", index=False)
per_image.to_csv(OUT_DIR / "T1_per_image.csv", index=False)
print(summary.to_string(index=False))
print("\non_gt_destroyed / on_gt_damaged_or_destroyed: share of hot spots whose zone touches ground-truth destroyed "
      "(damaged or destroyed) pixels, as in v1. point_on_gt_*: the same for the marked point itself, within "
      f"{POINT_RADIUS_PX} px. safe_on_gt_building: share of safe areas that sit on a ground-truth building.")

Images:   0%|          | 0/645 [00:00<?, ?it/s]

   images_from source  images  hot_spots  on_gt_destroyed  ceiling_from_gt_mask  on_gt_damaged_or_destroyed  point_on_gt_destroyed  point_on_gt_damage  safe_on_gt_building
           all    all     645        694            0.349                 0.466                       0.778                  0.282               0.736                0.028
           all  other      85         48            0.500                 0.432                       0.625                  0.417               0.479                0.167
           all    xBD     560        646            0.337                 0.470                       0.789                  0.272               0.755                0.017
with destroyed    all     216        397            0.610                 0.732                       0.897                  0.494               0.846                0.021
with destroyed  other      22         31            0.774                 0.704                       0.871                  0.645          

## 2. Overlays to check by eye, and v1 against v2
The same 20 images as v1: the 10 xBD and 10 other images with the most ground-truth destroyed pixels. Each figure has four panels:
1. the after-image, or on a PC the true damage map, with all markers;
2. danger;
3. safety, with the 3 safest open areas as green triangles;
4. search priority, with the top 3 hot spots as numbered cyan circles.

The colour scale is the same for every image (`DISPLAY_MAX`), so images can be compared.

In [4]:
# ── Cell 4: Overlays for the 20 images to check by eye, v1 against v2, README and config ──
COLOURS = np.array([[45, 45, 45], [60, 180, 75], [255, 200, 0], [230, 25, 75]], np.uint8)   # the true damage map

def find_image(name):
    """The after-image in the mirror (same candidates as the D4 notebooks' resolve_path), or None on a PC."""
    if DATA_ROOT is None:
        return None
    for c in (DATA_ROOT / "train_images" / name, DATA_ROOT / "train_images" / "train_images" / name):
        if c.exists():
            return c
    return None

def overlay(rec):
    name, stem = rec["image"], Path(rec["image"]).stem
    pred, gt = read_mask(PRED_DIR / name), read_mask(GT_DIR / name)
    danger, safety, priority, _ = survivability_maps(pred)
    path = find_image(name)
    if path:
        first = background = np.array(Image.open(path).convert("RGB"))
        first_title = "After image + all markers"
    else:                                   # no after-image here: the true damage map, with the markers on it
        first, background = COLOURS[gt], np.full((*pred.shape, 3), 128, np.uint8)
        first_title = "True damage map + all markers (red = destroyed)"
    fig, axes = plt.subplots(1, 4, figsize=(20, 5.4))
    panels = [(first_title, first, None, None, None),
              ("Danger (kernel density)", background, danger, "inferno", DISPLAY_MAX),
              ("Safety + 3 safest open areas", background, safety, "viridis", 1.0),
              ("Search priority + top 3 hot spots", background, priority, "inferno", DISPLAY_MAX)]
    for ax, (title, base, m, cmap, vmax) in zip(axes, panels):
        ax.imshow(base)
        if m is not None:
            shown = ax.imshow(m, cmap=cmap, alpha=0.55, vmin=0, vmax=vmax)
            fig.colorbar(shown, ax=ax, fraction=0.046, pad=0.02)
        ax.set_title(title)
        ax.axis("off")
    for ax in (axes[0], axes[2]):
        for a in rec["safe_areas"]:
            ax.plot(*a["xy"], "^", ms=12, color="lime", mec="black")
    for ax in (axes[0], axes[3]):
        for s in rec["hot_spots"]:
            x, y = s["point_xy"]
            ax.plot(x, y, "o", ms=16, mfc="none", mec="cyan", mew=2)
            ax.text(x + 14, y, str(s["rank"]), color="cyan", fontsize=13, weight="bold")
    fig.suptitle(f"{stem}  ({rec['event']}, {rec['source']})")
    fig.tight_layout()
    fig.savefig(OUT_DIR / "overlays" / f"{stem}.jpg", dpi=80)
    plt.close(fig)
    return path is not None

check = pd.concat([g.nlargest(N_CHECK, "gt_destroyed_px") for _, g in per_image.groupby("source")])
by_name = {r["image"]: r for r in records}
found = [overlay(by_name[n]) for n in tqdm(check["image"], desc="Overlays")]
if not all(found):
    print(f"Note: {found.count(False)} of {len(found)} after-images are not on this machine, "
          "so panel 1 shows the true damage map instead")

# The 20 checked images: v1 (19 Oct check) against v2, with the same tests
safe_on_20 = [a for n in check["image"] for a in by_name[n]["safe_areas"]]
on20 = {"circles": int(check["hot_spots"].sum()),
        "point_on_destroyed": int(check["point_on_gt_destroyed"].sum()),
        "point_on_damage": int(check["point_on_gt_damage"].sum()),
        "safe_areas": len(safe_on_20),
        "safe_within_50px_of_edge": sum(a["edge_px"] <= 50 for a in safe_on_20),
        "safe_on_building": sum(a["on_gt_building"] for a in safe_on_20)}
compare = pd.DataFrame([{"on the 20 checked images": k, "v1": V1_ON_20[k], "v2": on20[k]} for k in V1_ON_20])
print(compare.to_string(index=False))

def as_markdown(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:              # `tabulate` missing: fall back to a plain-text table
        return "```\n" + df.to_string(index=False) + "\n```"

examples = pd.concat([g.head(3) for _, g in check.groupby("source")])
config = {"task": "T1 v2", "smoke_test": SMOKE_TEST, "ran_on": "Kaggle" if ON_KAGGLE else "PC",
          "after_images": DATA_ROOT is not None, "predictions": str(PRED_DIR), "images": len(names),
          "sigma_px": SIGMA_PX, "damaged_weight": DAMAGED_WEIGHT, "safe_scale_px": SAFE_SCALE_PX,
          "hot_threshold": HOT_THRESHOLD, "min_spot_px": MIN_SPOT_PX, "safe_min_gap_px": SAFE_MIN_GAP_PX,
          "safe_edge_margin_px": SAFE_EDGE_MARGIN_PX, "point_radius_px": POINT_RADIUS_PX,
          "display_max": DISPLAY_MAX, "run_time_min": round((time.time() - t_start) / 60, 1)}
(OUT_DIR / "config.json").write_text(json.dumps(config, indent=1), encoding="utf-8")
(OUT_DIR / "README.md").write_text("\n".join([
    "# T1 v2 survivability heatmap", "",
    f"{len(names)} validation images{' (SMOKE TEST)' if SMOKE_TEST else ''}, maps from R9's predicted masks "
    f"(`{PRED_SET}`), run on {'Kaggle' if ON_KAGGLE else 'a PC'}. The maps are the same as v1's. Three fixes: each "
    f"hot spot is marked at its strongest point, zones under {MIN_SPOT_PX} px are ignored, and safe areas stay "
    f"{SAFE_EDGE_MARGIN_PX} px from the image edge.", "",
    "## The 20 checked images: v1 against v2", "",
    as_markdown(compare), "",
    f"- `point_on_destroyed`, `point_on_damage`: hot spots whose marked point is within {POINT_RADIUS_PX} px of a "
    "ground-truth destroyed (damaged or destroyed) building. v1's numbers come from the 19 Oct check of its "
    "overlays, with the same tests.",
    "- `safe_within_50px_of_edge`, `safe_on_building`: safe areas on the image edge, and on a ground-truth building.", "",
    "## Hot spots against the ground truth (all images)", "",
    as_markdown(summary), "",
    "- `on_gt_destroyed`, `on_gt_damaged_or_destroyed`: share of hot spots whose zone touches ground-truth destroyed "
    "(damaged or destroyed) pixels, as in v1. Read `on_gt_destroyed` against `ceiling_from_gt_mask`, the same share "
    "when the maps come from the ground-truth mask.",
    f"- `point_on_gt_destroyed`, `point_on_gt_damage`: the same for the marked point itself (within {POINT_RADIUS_PX} px).",
    "- `safe_on_gt_building`: share of safe areas that sit on a ground-truth building.", "",
    "## The 20 images to check by eye", "",
    as_markdown(check[["image", "source", "gt_destroyed_px", "hot_spots", "point_on_gt_destroyed",
                       "point_on_gt_damage", "safe_on_gt_building"]]), "",
    "## Examples (3 xBD, 3 other)", "",
    *[f"![{n}](overlays/{Path(n).stem}.jpg)\n" for n in examples["image"]],
    "## Files", "",
    "- `hotspots.json`: per image, the top 3 hot spots (marked point, area, peak, compass position, ground-truth "
    "checks) and the 3 safest open areas (point, distance to damage, distance to the edge, compass position, "
    "ground-truth check)",
    "- `maps/<name>_{kde,safe,priority}.png`: 256 x 256, value x 255",
    "- `overlays/<name>.jpg`, `T1_summary.csv`, `T1_per_image.csv`, `config.json`",
]), encoding="utf-8")
print(f"✓ Wrote {len(check)} overlays, README.md and config.json ({(time.time() - t_start) / 60:.1f} min)")

Overlays:   0%|          | 0/20 [00:00<?, ?it/s]

Note: 20 of 20 after-images are not on this machine, so panel 1 shows the true damage map instead
on the 20 checked images  v1  v2
                 circles  48  46
      point_on_destroyed  26  31
         point_on_damage  36  39
              safe_areas  57  57
safe_within_50px_of_edge  52   0
        safe_on_building   9   4
✓ Wrote 20 overlays, README.md and config.json (7.5 min)
